In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf


def fetch_market_data(tickers, start_date, end_date):
    print(f"Data for {tickers}")
    data = yf.download(tickers, start=start_date, end=end_date)
    return data


In [ ]:
def process_risk_and_liquidity(data, ticker):
    """Calculate Log Returns, Rolling Volatility, Amihud Liquidity Ratio, and Parametric VaR."""

    df = pd.DataFrame()
    df["Close"] = data["Close"][ticker] if isinstance(data.columns, pd.MultiIndex) else data["Close"]
    df["Volume"] = data["Volume"][ticker] if isinstance(data.columns, pd.MultiIndex) else data["Volume"]
    # df["Close"] = data["Close"][ticker]
    # df["Volume"] = data["Volume"][ticker]


    df = df.dropna().copy()

    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

    #Daily USD Dollar Volume (Price * Volume)
    df["Dollar_Volume"] = df["Close"] * df["Volume"]

    #Amihud Liquidity Ratio = |Return| / Dollar Volume
    #(Scaled by 1e11 for high-volume US benchmarks like SPY)
    df["Amihud_Liquidity_Impact"] = (
        df["Log_Return"].abs() / df["Dollar_Volume"]
    ) * 1e11

    #Rolling Realized Volatility (Annualized 10-day & 30-day windows)
    #Trading days = 252
    df["Vol_10D_Ann"] = (
        df["Log_Return"].rolling(window=10).std() * np.sqrt(252) * 100
    )
    df["Vol_30D_Ann"] = (
        df["Log_Return"].rolling(window=30).std() * np.sqrt(252) * 100
    )

    portfolio_value = 100000
    rolling_std = df["Log_Return"].rolling(window=30).std()

    df["VaR_95_USD"] = portfolio_value * (1.645 * rolling_std)
    df["VaR_99_USD"] = portfolio_value * (2.326 * rolling_std)

    df = df.dropna().copy()

    vol_threshold = 18.0
    liquidity_85th_percentile = df["Amihud_Liquidity_Impact"].quantile(0.85)

    # Risk Flag: High Volatility AND High Liquidity Impact
    df["Risk_Alert_Flag"] = np.where(
        (df["Vol_30D_Ann"] > vol_threshold)
        & (df["Amihud_Liquidity_Impact"] > liquidity_85th_percentile),
        "CRITICAL STRESS",
        "NORMAL",
    )

    # Product Action Recommendation for Traders
    df["Product_Action_Recommendation"] = np.where(
        df["Risk_Alert_Flag"] == "CRITICAL STRESS",
        "Reduce Order Size / Divert to Dark Pool",
        "Standard Execution",
    )

    df = df.round(8)

    df["Ticker"] = ticker

    return df


In [ ]:
if __name__ == "__main__":
    TICKER = "SPY"
    START_DATE = "2024-01-01"
    END_DATE = "2026-08-01"

    raw_data = fetch_market_data(TICKER, START_DATE, END_DATE)

    spy_df = process_risk_and_liquidity(raw_data, TICKER)

    # Save to CSV
    output_filename = "us_capital_markets_data.csv"
    spy_df.to_csv(output_filename)

    print(f"Data saved to '{output_filename}'")
    print(
        spy_df[
            [
                "Close",
                "Log_Return",
                "Vol_30D_Ann",
                "Amihud_Liquidity_Impact",
                "VaR_95_USD",
            ]
        ].head()
    )

/tmp/ipykernel_1293/3645391586.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

Data for SPY
Data saved to 'us_capital_markets_data.csv'
                 Close  Log_Return  Vol_30D_Ann  Amihud_Liquidity_Impact  \
Date                                                                       
2024-02-14  483.970215    0.009047    11.615281                 0.027333   
2024-02-15  487.309479    0.006876    11.307463                 0.022875   
2024-02-16  484.882690   -0.004992    11.393483                 0.013631   
2024-02-20  482.213135   -0.005521    11.611278                 0.015960   
2024-02-21  482.649994    0.000906    11.023629                 0.003164   

             VaR_95_USD  
Date                     
2024-02-14  1203.636533  
2024-02-15  1171.738805  
2024-02-16  1180.652642  
2024-02-20  1203.221748  
2024-02-21  1142.326388  
